# Clasificación Binaria con InternVL 2.5 — Entrada: imagen + texto

Este notebook utiliza **InternVL 2.5 (26B)**, un modelo multimodal state-of-the-art, para realizar clasificación binaria (YES/NO) sobre memes, determinando si contienen contenido sexista.

## 1. Importar librerías

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import torch
import json
import os
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    precision_recall_curve,
    roc_curve,
    auc,
    classification_report
)

from pyevall.evaluation import PyEvALLEvaluation
from pyevall.metrics.metricfactory import MetricFactory

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuración y parámetros

In [ ]:
# ── Token de Hugging Face (necesario para algunos modelos) ──────────────────
os.environ["HF_TOKEN"] = ""  # Opcional: añade tu token si es necesario

# ── Modelo ───────────────────────────────────────────────────────────────────
MODEL_NAME = "OpenGVLab/InternVL2_5-26B"  # Modelo state-of-the-art de 26B parámetros

# ── Paths ────────────────────────────────────────────────────────────────────
MAIN_PATH  = ".."  # Carpeta padre del proyecto
GROUP_ID   = "BeingChillingWeWillWin"
MODEL_ID   = "internvl25_26b"

LABEL_COLUMN = "label"

DATA_TRAIN_PATH = os.path.join(MAIN_PATH, "preprocessed_data", "train_split.json")
DATA_VAL_PATH   = os.path.join(MAIN_PATH, "preprocessed_data", "dev_split.json")
DATA_TEST_PATH  = os.path.join(MAIN_PATH, "preprocessed_data", "test_split.json")

DATA_BASE_DIR   = os.path.join(MAIN_PATH, "materials", "dataset_task2_exist2026")

PREDICTIONS_DIR = os.path.join(MAIN_PATH, "predictions")
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

# ── Hiperparámetros de inferencia ────────────────────────────────────────────
BATCH_SIZE    = 1       # InternVL 2.5 es grande, procesar una imagen a la vez
MAX_NEW_TOKENS = 512    # Tokens máximos de respuesta
TEMPERATURE   = 0.3     # Baja temperatura para respuestas más determinísticas
TOP_P         = 0.9

# ── Configuración del dispositivo ────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# Usar precisión reducida para modelos grandes (bf16 si disponible)
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"Using dtype: {DTYPE}")

# ── Mapeo de etiquetas ───────────────────────────────────────────────────────
label_map         = {"NO": 0, "YES": 1}
label_map_inverse = {0: "NO", 1: "YES"}

## 3. Carga y preprocesamiento de datos

In [ ]:
def load_json_dataset(path):
    """Carga el JSON orientado a diccionario y devuelve un DataFrame."""
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return pd.DataFrame(data.values())

# Cargar splits
train_df = load_json_dataset(DATA_TRAIN_PATH)
val_df   = load_json_dataset(DATA_VAL_PATH)
test_df  = load_json_dataset(DATA_TEST_PATH)

# Mapear etiquetas a enteros
train_df["label_int"] = train_df[LABEL_COLUMN].map(label_map)
val_df["label_int"]   = val_df[LABEL_COLUMN].map(label_map)
test_df["label_int"]  = -1  # Test no tiene etiquetas

print(f"Train size: {len(train_df)} | Val size: {len(val_df)} | Test size: {len(test_df)}")
print(f"\nEjemplo de ruta de imagen: {train_df['path_memes'].iloc[0]}")
print(f"\nDistribución de etiquetas en TRAIN:")
print(train_df[LABEL_COLUMN].value_counts())
print(f"\nDistribución de etiquetas en VAL:")
print(val_df[LABEL_COLUMN].value_counts())

## 4. Carga del modelo InternVL 2.5

**InternVL 2.5** es un modelo multimodal de gran escala (26B parámetros) que combina:
- Un encoder visual potente (InternViT-6B)
- Un LLM avanzado para razonamiento multimodal
- Capacidades de chat conversacional

**Características clave:**
- Soporta imágenes de alta resolución (hasta 4K)
- Excelente en tareas de razonamiento visual
- Formato de conversación multi-turno
- Requiere GPU con mínimo 40GB VRAM (recomendado: A100)

In [ ]:
print("Cargando InternVL 2.5 (26B)...")
print("Esto puede tardar varios minutos dependiendo de tu conexión...\n")

# ── Cargar modelo con cuantización 4-bit para reducir uso de memoria ─────────
# Si tienes GPU con >40GB VRAM, puedes remover la cuantización para mejor rendimiento
from transformers import BitsAndBytesConfig

# Configuración de cuantización 4-bit con NF4 (recomendado para modelos grandes)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=DTYPE,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Cargar el modelo
# NOTA: Si no tienes suficiente VRAM, considera usar InternVL2_5-8B en lugar del 26B
model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    quantization_config=quantization_config,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
).eval()

# Cargar el tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

print(f"✓ Modelo cargado exitosamente en {DEVICE}")
print(f"✓ Usando cuantización 4-bit para optimizar memoria")

## 5. Definición del prompt para clasificación binaria

InternVL 2.5 usa un formato de chat conversacional. Diseñamos un prompt específico para la tarea de detección de sexismo.

In [ ]:
def build_classification_prompt(text_in_meme):
    """
    Construye el prompt para clasificación binaria de sexismo.
    
    InternVL 2.5 funciona mejor con instrucciones claras y específicas.
    El formato es conversacional con roles 'user' y 'assistant'.
    """
    system_instruction = (
        "You are an expert content moderator specialized in identifying sexist content in memes. "
        "Your task is to analyze both the visual content and the text to determine if the meme "
        "contains sexist elements (stereotypes, discrimination, objectification, or derogatory content "
        "towards any gender).\n\n"
    )
    
    user_message = (
        f"Analyze this meme carefully. The text in the image says: \"{text_in_meme}\"\n\n"
        "Consider both the visual elements and the text. Does this meme contain sexist content?\n\n"
        "Answer ONLY with 'YES' if it contains sexist content, or 'NO' if it doesn't. "
        "Then provide a brief explanation (1-2 sentences) of your reasoning.\n\n"
        "Format your response as:\n"
        "CLASSIFICATION: [YES/NO]\n"
        "REASON: [your explanation]"
    )
    
    # Formato esperado por InternVL 2.5
    prompt = system_instruction + user_message
    return prompt

# Ejemplo de prompt
example_text = "Las mujeres al volante..."
example_prompt = build_classification_prompt(example_text)
print("═" * 80)
print("EJEMPLO DE PROMPT:")
print("═" * 80)
print(example_prompt)
print("═" * 80)

## 6. Función de inferencia

Esta función procesa una imagen con InternVL 2.5 y extrae la clasificación binaria.

In [ ]:
def parse_model_response(response_text):
    """
    Parsea la respuesta del modelo para extraer la clasificación YES/NO.
    
    Returns:
        tuple: (classification: str, reason: str, confidence: float)
    """
    response_upper = response_text.upper()
    
    # Buscar la clasificación explícita
    if "CLASSIFICATION: YES" in response_upper or "CLASSIFICATION:YES" in response_upper:
        classification = "YES"
    elif "CLASSIFICATION: NO" in response_upper or "CLASSIFICATION:NO" in response_upper:
        classification = "NO"
    # Fallback: buscar YES/NO en el texto
    elif response_upper.strip().startswith("YES"):
        classification = "YES"
    elif response_upper.strip().startswith("NO"):
        classification = "NO"
    else:
        # Si no podemos parsear, asignamos confianza baja
        classification = "NO"  # Default conservador
        print(f"[WARN] No se pudo parsear la respuesta: {response_text[:100]}")
    
    # Extraer razón si está presente
    reason = ""
    if "REASON:" in response_text:
        reason = response_text.split("REASON:", 1)[1].strip()
    else:
        reason = response_text
    
    # Confianza heurística: 0.9 si encuentra el formato esperado, 0.6 si no
    confidence = 0.9 if ("CLASSIFICATION:" in response_text) else 0.6
    
    return classification, reason, confidence


@torch.no_grad()
def classify_image(image_path, text_in_meme, model, tokenizer):
    """
    Clasifica una imagen usando InternVL 2.5.
    
    Args:
        image_path (str): Ruta a la imagen
        text_in_meme (str): Texto que aparece en el meme
        model: Modelo InternVL cargado
        tokenizer: Tokenizer correspondiente
    
    Returns:
        dict: {
            'classification': 'YES' or 'NO',
            'confidence': float,
            'reason': str,
            'raw_response': str
        }
    """
    try:
        # Cargar y preparar la imagen
        image = Image.open(image_path).convert('RGB')
        
        # Construir el prompt
        prompt = build_classification_prompt(text_in_meme)
        
        # InternVL 2.5 espera un formato específico para el chat
        # Usar el método chat() del modelo que maneja automáticamente el formato
        generation_config = dict(
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True if TEMPERATURE > 0 else False,
        )
        
        # Generar respuesta
        response = model.chat(
            tokenizer=tokenizer,
            pixel_values=None,  # Se procesará internamente desde la imagen
            image=image,
            msgs=[{'role': 'user', 'content': prompt}],
            generation_config=generation_config,
        )
        
        # Parsear la respuesta
        classification, reason, confidence = parse_model_response(response)
        
        return {
            'classification': classification,
            'confidence': confidence,
            'reason': reason,
            'raw_response': response
        }
        
    except Exception as e:
        print(f"[ERROR] Error procesando imagen {image_path}: {e}")
        return {
            'classification': 'NO',
            'confidence': 0.0,
            'reason': f'Error: {str(e)}',
            'raw_response': ''
        }

print("✓ Función de inferencia definida")

## 7. Inferencia sobre el conjunto de validación (DEV)

Procesamos el conjunto de validación para evaluar el rendimiento del modelo.

In [ ]:
def process_dataset(df, base_dir, model, tokenizer, split_name="dev"):
    """
    Procesa un dataset completo con InternVL 2.5.
    
    Returns:
        list: Lista de diccionarios con predicciones
    """
    results = []
    base_path = Path(base_dir)
    
    print(f"\nProcesando {len(df)} imágenes del conjunto {split_name.upper()}...\n")
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Inferencia {split_name}"):
        img_path = base_path / row['path_memes']
        text_in_meme = str(row.get('text', '') or '')
        
        # Clasificar la imagen
        prediction = classify_image(img_path, text_in_meme, model, tokenizer)
        
        # Guardar resultado
        result = {
            'id_EXIST': str(row['id_EXIST']),
            'classification': prediction['classification'],
            'confidence': prediction['confidence'],
            'reason': prediction['reason'],
            'raw_response': prediction['raw_response'],
        }
        
        # Si tenemos la etiqueta real (dev/train), añadirla
        if 'label_int' in row.index and row['label_int'] >= 0:
            result['true_label'] = label_map_inverse[row['label_int']]
        
        results.append(result)
    
    return results

# Procesar conjunto de validación
val_results = process_dataset(val_df, DATA_BASE_DIR, model, tokenizer, split_name="dev")

print(f"\n✓ {len(val_results)} predicciones completadas para DEV")

## 8. Evaluación de resultados en DEV

In [ ]:
# Convertir a formato numérico para métricas
y_pred_labels = [label_map[r['classification']] for r in val_results]
y_true_labels = [label_map[r['true_label']] for r in val_results]
confidences = [r['confidence'] for r in val_results]

# ── Métricas básicas ─────────────────────────────────────────────────────────
accuracy = accuracy_score(y_true_labels, y_pred_labels)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true_labels, y_pred_labels, average='macro', zero_division=0
)

print("\n" + "═" * 80)
print(" RESULTADOS EN VALIDACIÓN (DEV)")
print("═" * 80)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f} (macro)")
print(f"Recall:    {recall:.4f} (macro)")
print(f"F1-Score:  {f1:.4f} (macro)")
print("═" * 80)

# ── Reporte detallado ────────────────────────────────────────────────────────
print("\nClassification Report:")
print(classification_report(y_true_labels, y_pred_labels, target_names=['NO', 'YES']))

# ── Distribución de predicciones ─────────────────────────────────────────────
pred_df = pd.DataFrame({
    'predicted': [label_map_inverse[p] for p in y_pred_labels],
    'true': [label_map_inverse[t] for t in y_true_labels]
})

print("\nDistribución de predicciones:")
print(pred_df['predicted'].value_counts())

# ── Matriz de confusión ──────────────────────────────────────────────────────
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_true_labels, y_pred_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['NO', 'YES'], yticklabels=['NO', 'YES'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'Confusion Matrix - InternVL 2.5 on DEV\nAccuracy: {accuracy:.4f} | F1: {f1:.4f}')
plt.tight_layout()
plt.show()

## 9. Análisis de confianza

In [ ]:
# Analizar la confianza de las predicciones
pred_analysis_df = pd.DataFrame({
    'confidence': confidences,
    'correct': [p == t for p, t in zip(y_pred_labels, y_true_labels)]
})

# Métricas por nivel de confianza
print("\nAnálisis de confianza:")
print(f"Confianza promedio: {np.mean(confidences):.4f}")
print(f"Confianza std:      {np.std(confidences):.4f}")
print(f"\nPrecisiones correctas con alta confianza (>0.8): "
      f"{pred_analysis_df[pred_analysis_df['confidence'] > 0.8]['correct'].mean():.4f}")
print(f"Predicciones correctas con baja confianza (<0.7): "
      f"{pred_analysis_df[pred_analysis_df['confidence'] < 0.7]['correct'].mean():.4f}")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de confianzas
axes[0].hist(confidences, bins=20, edgecolor='black', alpha=0.7)
axes[0].axvline(np.mean(confidences), color='red', linestyle='--', label=f'Mean: {np.mean(confidences):.3f}')
axes[0].set_xlabel('Confidence')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Prediction Confidence')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Precisión vs confianza
conf_bins = pd.cut(confidences, bins=10)
acc_by_conf = pred_analysis_df.groupby(conf_bins)['correct'].mean()
bin_centers = [interval.mid for interval in acc_by_conf.index]
axes[1].plot(bin_centers, acc_by_conf.values, marker='o', linewidth=2)
axes[1].set_xlabel('Confidence (binned)')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy vs Confidence')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Evaluación con PyEvALL (formato oficial de la competencia)

In [ ]:
# Formato PyEvALL para predicciones
dev_preds_for_pyevall = [
    {'test_case': 'EXIST2025', 'id': r['id_EXIST'], 'value': r['classification']}
    for r in val_results
]
dev_preds_df = pd.DataFrame(dev_preds_for_pyevall)
dev_preds_path = os.path.join(PREDICTIONS_DIR, 'dev_predictions_temp_internvl.json')
with open(dev_preds_path, 'w', encoding='utf-8') as f:
    f.write(dev_preds_df.to_json(orient='records'))

# Formato gold
dev_gold = [
    {'test_case': 'EXIST2025', 'id': str(id_exist), 'value': label}
    for id_exist, label in zip(val_df['id_EXIST'].values, val_df[LABEL_COLUMN].values)
]
dev_gold_df = pd.DataFrame(dev_gold)
dev_gold_path = os.path.join(PREDICTIONS_DIR, 'dev_gold_temp_internvl.json')
with open(dev_gold_path, 'w', encoding='utf-8') as f:
    f.write(dev_gold_df.to_json(orient='records'))

# Evaluar con PyEvALL
test_eval = PyEvALLEvaluation()
metrics = [MetricFactory.Accuracy.value, MetricFactory.FMeasure.value]
report = test_eval.evaluate(dev_preds_path, dev_gold_path, metrics)

print("\n" + "═" * 80)
print(" EVALUACIÓN OFICIAL CON PyEvALL (DEV)")
print("═" * 80)
report.print_report()
print("═" * 80)

## 11. Guardar predicciones detalladas de DEV

In [ ]:
# Guardar resultados completos con explicaciones
dev_detailed_path = os.path.join(PREDICTIONS_DIR, f'{GROUP_ID}_{MODEL_ID}_dev_detailed.json')
with open(dev_detailed_path, 'w', encoding='utf-8') as f:
    json.dump(val_results, f, ensure_ascii=False, indent=2)

print(f"✓ Predicciones detalladas de DEV guardadas en: {dev_detailed_path}")

# Guardar también en formato de probabilidades (confianza como proxy)
dev_probs = [
    {
        'id': r['id_EXIST'],
        'prob_YES': r['confidence'] if r['classification'] == 'YES' else (1 - r['confidence']),
        'label': r.get('true_label', 'UNKNOWN')
    }
    for r in val_results
]
dev_probs_path = os.path.join(PREDICTIONS_DIR, f'{GROUP_ID}_{MODEL_ID}_probs_dev.json')
with open(dev_probs_path, 'w', encoding='utf-8') as f:
    json.dump(dev_probs, f, ensure_ascii=False, indent=2)

print(f"✓ Probabilidades de DEV guardadas en: {dev_probs_path}")

## 12. Inferencia en TEST y generación de predicciones finales

In [ ]:
# Procesar conjunto de test
test_results = process_dataset(test_df, DATA_BASE_DIR, model, tokenizer, split_name="test")

print(f"\n✓ {len(test_results)} predicciones completadas para TEST")

# Distribución de predicciones
test_preds_series = pd.Series([r['classification'] for r in test_results])
print("\nDistribución de predicciones en TEST:")
print(test_preds_series.value_counts())
print(f"\nPorcentaje de YES: {100 * (test_preds_series == 'YES').mean():.2f}%")
print(f"Porcentaje de NO:  {100 * (test_preds_series == 'NO').mean():.2f}%")

## 13. Guardar predicciones finales para TEST

In [ ]:
# ── Formato oficial para la competencia (PyEvALL) ────────────────────────────
test_preds_for_submission = [
    {'test_case': 'EXIST2025', 'id': r['id_EXIST'], 'value': r['classification']}
    for r in test_results
]
test_preds_df = pd.DataFrame(test_preds_for_submission)

output_filename = f"{GROUP_ID}_{MODEL_ID}.json"
output_path = os.path.join(PREDICTIONS_DIR, output_filename)

with open(output_path, 'w', encoding='utf-8') as f:
    f.write(test_preds_df.to_json(orient='records'))

print(f"\n✓ Predicciones oficiales guardadas en: {output_path}")

# ── Guardar resultados detallados con explicaciones ───────────────────────────
test_detailed_path = os.path.join(PREDICTIONS_DIR, f'{GROUP_ID}_{MODEL_ID}_test_detailed.json')
with open(test_detailed_path, 'w', encoding='utf-8') as f:
    json.dump(test_results, f, ensure_ascii=False, indent=2)

print(f"✓ Predicciones detalladas de TEST guardadas en: {test_detailed_path}")

# ── Guardar probabilidades (confianza como proxy) ─────────────────────────────
test_probs = [
    {
        'id': r['id_EXIST'],
        'prob_YES': r['confidence'] if r['classification'] == 'YES' else (1 - r['confidence'])
    }
    for r in test_results
]
test_probs_path = os.path.join(PREDICTIONS_DIR, f'{GROUP_ID}_{MODEL_ID}_probs_test.json')
with open(test_probs_path, 'w', encoding='utf-8') as f:
    json.dump(test_probs, f, ensure_ascii=False, indent=2)

print(f"✓ Probabilidades de TEST guardadas en: {test_probs_path}")

## 14. Análisis cualitativo: ejemplos de predicciones

In [ ]:
# Mostrar algunos ejemplos del conjunto de validación
print("\n" + "═" * 80)
print(" EJEMPLOS DE PREDICCIONES (DEV)")
print("═" * 80)

# Seleccionar ejemplos interesantes
np.random.seed(42)
sample_indices = np.random.choice(len(val_results), min(5, len(val_results)), replace=False)

for i, idx in enumerate(sample_indices, 1):
    result = val_results[idx]
    print(f"\n{'─' * 80}")
    print(f"EJEMPLO {i}")
    print(f"{'─' * 80}")
    print(f"ID: {result['id_EXIST']}")
    print(f"Predicción: {result['classification']} (confianza: {result['confidence']:.3f})")
    print(f"Verdadero:  {result.get('true_label', 'N/A')}")
    print(f"\nRazón del modelo:")
    print(f"  {result['reason'][:200]}..." if len(result['reason']) > 200 else f"  {result['reason']}")
    print(f"\nRespuesta completa:")
    print(f"  {result['raw_response'][:300]}..." if len(result['raw_response']) > 300 else f"  {result['raw_response']}")

print("\n" + "═" * 80)

## 15. Resumen final

In [ ]:
print("\n" + "═" * 80)
print(" RESUMEN FINAL - InternVL 2.5 (26B)")
print("═" * 80)
print(f"\n📊 MÉTRICAS EN VALIDACIÓN (DEV):")
print(f"   Accuracy:  {accuracy:.4f}")
print(f"   Precision: {precision:.4f} (macro)")
print(f"   Recall:    {recall:.4f} (macro)")
print(f"   F1-Score:  {f1:.4f} (macro)")
print(f"\n📁 ARCHIVOS GENERADOS:")
print(f"   • Predicciones oficiales (TEST): {output_filename}")
print(f"   • Predicciones detalladas (DEV):  {GROUP_ID}_{MODEL_ID}_dev_detailed.json")
print(f"   • Predicciones detalladas (TEST): {GROUP_ID}_{MODEL_ID}_test_detailed.json")
print(f"   • Probabilidades (DEV):           {GROUP_ID}_{MODEL_ID}_probs_dev.json")
print(f"   • Probabilidades (TEST):          {GROUP_ID}_{MODEL_ID}_probs_test.json")
print(f"\n💡 NOTAS:")
print(f"   • Modelo usado: {MODEL_NAME}")
print(f"   • Cuantización: 4-bit NF4")
print(f"   • Temperature: {TEMPERATURE}")
print(f"   • Total imágenes procesadas: {len(val_results) + len(test_results)}")
print("\n" + "═" * 80)
print("✓ Pipeline completado exitosamente")
print("═" * 80)